**Import various packages**

In [ ]:
import sys
sys.path.insert(0, 'Utilities')
sys.path.append('../input') 
import time
import torch
import imageio
import scipy.io
import numpy as np
import torch.nn as nn
from scipy import linalg
import scipy.sparse as sp
import torch.optim as optim
from numpy.linalg import norm
from scipy.sparse import diags
import matplotlib.pyplot as plt
from numpy import inner, conjugate
from scipy.sparse import coo_matrix
from torch.nn import functional as F
from scipy.sparse import random, identity
from torch.optim.lr_scheduler import StepLR
from scipy.sparse.linalg import spsolve, splu
from scipy.sparse.linalg._isolve.utils import make_system
from scipy.sparse import lil_matrix, csr_matrix, csc_matrix
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

**Generation functions of the impedance matrix and source term**

In [ ]:
def matrix3fd4(nx, ny, nz, dlx, dly, dlz, f, delta, c_vec):
    # imaginary unit
    ii = 1j

    # problem scale
    N = (nx - 2) * (ny - 2) * (nz - 2)

    # nonzero elements of impedance matrix
    if nx < 6 or ny < 6 or nz < 6:
        spn = N * N
    else:
        spn = 14 * (nx - 6) * (ny - 6) * (nz - 6) + \
              12 * 2 * ((nx - 6) * (ny - 6) + (ny - 6) * (nz - 6) + (nx - 6) * (nz - 6)) + \
              11 * 2 * ((nx - 6) * (ny - 6) + (ny - 6) * (nz - 6) + (nx - 6) + (nz - 6)) + \
              11 * 2 * 4 * ((nx - 6) + (ny - 6) + (nz - 6)) + \
              9 * 2 * 4 * ((nx - 6) + (ny - 6) + (nz - 6)) + \
              8 * ((13 - 6) + 3 * (13 - 5) + 3 * (13 - 4) + (13 - 3))
    #print(spn)
    # sparse store: vector space
    ai = np.zeros(spn, dtype=int)
    aj = np.zeros(spn, dtype=int)
    as_ = np.zeros(spn, dtype=complex)
    pa = 0

    # set angular frequency
    omega = 2 * np.pi * f

    # PML parameter: the ratio of reflection
    R = 1e-3

    # stencil coefficients of 4-order FD
    coeff2 = np.array([-1 / 12, 4 / 3, -5 / 2, 4 / 3, -1 / 12])
    coeff1 = np.array([-1 / 12, 2 / 3, 0, -2 / 3, 1 / 12])

    # set values
    a2, a1, a0, a_1, a_2 = coeff2
    b2, b1, b0, b_1, b_2 = coeff1

    for n in range(N):
        # grid coordinate conversion: y-slice rule
        j = n // ((nx - 2) * (nz - 2))
        k = (n - (nx - 2) * (nz - 2) * j) // (nx - 2)
        i = n - (nx - 2) * k - (nx - 2) * (nz - 2) * j

        # set velocity
        c = c_vec[n]

        # set PML attenuation function
        if i < delta - 1:
            dx = -3 * c / (2 * delta * dlx) * np.log(R) * ((delta - i - 1) / delta) ** 2
            dxp = -3 * c / (delta * dlx) ** 2 * np.log(R) * (-(delta - i - 1) / delta)
        elif i > nx - delta - 2:
            dx = -3 * c / (2 * delta * dlx) * np.log(R) * ((i - nx + delta + 2) / delta) ** 2
            dxp = -3 * c / (delta * dlx) ** 2 * np.log(R) * ((i - nx + delta + 2) / delta)
        else:
            dx = 0
            dxp = 0

        if j < delta - 1:
            dy = -3 * c / (2 * delta * dly) * np.log(R) * ((delta - j - 1) / delta) ** 2
            dyp = -3 * c / (delta * dly) ** 2 * np.log(R) * (-(delta - j - 1) / delta)
        elif j > ny - delta - 2:
            dy = -3 * c / (2 * delta * dly) * np.log(R) * ((j - ny + delta + 2) / delta) ** 2
            dyp = -3 * c / (delta * dly) ** 2 * np.log(R) * ((j - ny + delta + 2) / delta)
        else:
            dy = 0
            dyp = 0

        if k < delta - 1:
            dz = -3 * c / (2 * delta * dlz) * np.log(R) * ((delta - k - 1) / delta) ** 2
            dzp = -3 * c / (delta * dlz) ** 2 * np.log(R) * (-(delta - k - 1) / delta)
        elif k > nz - delta - 2:
            dz = -3 * c / (2 * delta * dlz) * np.log(R) * ((k - nz + delta + 2) / delta) ** 2
            dzp = -3 * c / (delta * dlz) ** 2 * np.log(R) * ((k - nz + delta + 2) / delta)
        else:
            dz = 0
            dzp = 0

        tx = 1 - ii * dx / omega
        ty = 1 - ii * dy / omega
        tz = 1 - ii * dz / omega

        nt = n + 1

        # set the nonzeros of matrix
        # left1
        if i != 0:
            ai[pa] = nt
            aj[pa] = nt - 1
            as_[pa] = a_1 / (tx ** 2) + b_1 * ii * dxp * dlx / (omega * tx ** 3)
            pa += 1

        # left2
        if i > 1:
            ai[pa] = nt
            aj[pa] = nt - 2
            as_[pa] = a_2 / (tx ** 2) + b_2 * ii * dxp * dlx / (omega * tx ** 3)
            pa += 1

        # right1
        if i != nx - 3:
            ai[pa] = nt
            aj[pa] = nt + 1
            as_[pa] = a1 / (tx ** 2) + b1 * ii * dxp * dlx / (omega * tx ** 3)
            pa += 1

        # right2
        if i < nx - 4:
            ai[pa] = nt
            aj[pa] = nt + 2
            as_[pa] = a2 / (tx ** 2) + b2 * ii * dxp * dlx / (omega * tx ** 3)
            pa += 1

        # up1
        if k != 0:
            ai[pa] = nt
            aj[pa] = nt - (nx - 2)
            as_[pa] = a_1 / (tz ** 2) * (dlx ** 2 / dlz ** 2) + b_1 * ii * dzp / (omega * tz ** 3) * (dlx ** 2 / dlz)
            pa += 1

        # up2
        if k > 1:
            ai[pa] = nt
            aj[pa] = nt - 2 * (nx - 2)
            as_[pa] = a_2 / (tz ** 2) * (dlx ** 2 / dlz ** 2) + b_2 * ii * dzp / (omega * tz ** 3) * (dlx ** 2 / dlz)
            pa += 1

        # down1
        if k != nz - 3:
            ai[pa] = nt
            aj[pa] = nt + (nx - 2)
            as_[pa] = a1 / (tz ** 2) * (dlx ** 2 / dlz ** 2) + b1 * ii * dzp / (omega * tz ** 3) * (dlx ** 2 / dlz)
            pa += 1

        # down2
        if k < nz - 4:
            ai[pa] = nt
            aj[pa] = nt + 2 * (nx - 2)
            as_[pa] = a2 / (tz ** 2) * (dlx ** 2 / dlz ** 2) + b2 * ii * dzp / (omega * tz ** 3) * (dlx ** 2 / dlz)
            pa += 1

        # back1
        if j != 0:
            ai[pa] = nt
            aj[pa] = nt - (nx - 2) * (nz - 2)
            as_[pa] = a_1 / (ty ** 2) * (dlx ** 2 / dly ** 2) + b_1 * ii * dyp / (omega * ty ** 3) * (dlx ** 2 / dly)
            pa += 1

        # back2
        if j > 1:
            ai[pa] = nt
            aj[pa] = nt - 2 * (nx - 2) * (nz - 2)
            as_[pa] = a_2 / (ty ** 2) * (dlx ** 2 / dly ** 2) + b_2 * ii * dyp / (omega * ty ** 3) * (dlx ** 2 / dly)
            pa += 1

        # front1
        if j != ny - 3:
            ai[pa] = nt
            aj[pa] = nt + (nx - 2) * (nz - 2)
            as_[pa] = a1 / (ty ** 2) * (dlx ** 2 / dly ** 2) + b1 * ii * dyp / (omega * ty ** 3) * (dlx ** 2 / dly)
            pa += 1

        # front2
        if j < ny - 4:
            ai[pa] = nt
            aj[pa] = nt + 2 * (nx - 2) * (nz - 2)
            as_[pa] = a2 / (ty ** 2) * (dlx ** 2 / dly ** 2) + b2 * ii * dyp / (omega * ty ** 3) * (dlx ** 2 / dly)
            pa += 1

        # inner
        ai[pa] = nt
        aj[pa] = nt
        if ny == 3:  # since y-slice, this return to 2D case
            as_[pa] = ((omega * dlx / c) ** 2 +
                       a0 * (1 / tx ** 2 + 1 / tz ** 2 * (dlx ** 2 / dlz ** 2)) +
                       b0 * ii / omega * (dlx * dxp / tx ** 3 + (dlx ** 2 / dlz) * dzp / tz ** 3))
        else:
            as_[pa] = ((omega * dlx / c) ** 2 +
                       a0 * (1 / tx ** 2 + 1 / ty ** 2 * (dlx ** 2 / dly ** 2) + 1 / tz ** 2 * (dlx ** 2 / dlz ** 2)) +
                       b0 * ii / omega * (dlx * dxp / tx ** 3 + (dlx ** 2 / dly) * dyp / ty ** 3 + (
                            dlx ** 2 / dlz) * dzp / tz ** 3))
        pa += 1

    # extract potential zeros
    mask = as_ != 0
    ai = ai[:pa][mask[:pa]]
    aj = aj[:pa][mask[:pa]]
    as_ = as_[:pa][mask[:pa]]

    # create sparse matrix
    C = sp.csr_matrix((as_, (ai-1, aj-1)), shape=(N, N))

    return C
    
def source_ofd(f, f0, N, h, c_vec, s_loc):
    # generate the right hand side term of source
    # last revision: 2022.7.6

    # initial unit
    ii = 1j

    # source location (count from 0)
    s0 = s_loc

    # source velocity
    sc = c_vec[s0][0]

    # Amplitude and phase
    t0 = 0.12
    Amp = 1e+5

    s = np.sqrt(2) * Amp / (np.pi * f0) * (f / f0) ** 2 * np.exp(-(f / f0) ** 2) * \
        sp.coo_matrix(([-h ** 2 / sc ** 2 * np.exp(-ii * 2 * np.pi * f * t0)], ([s0], [0])), shape=(N, 1))

    return s

**Basic parameter settings for the impendance matrix and source**

In [ ]:
nx = 66
ny = 66
nz = 66
# spatial step
h = 0.035
# dominant frequency of source
f = 20
f0 = 10
# numbers of PML layers
delta = 10
#c_vec is the wave velocity
cc = 4 * np.ones((nz - 2, nx - 2, ny - 2), dtype=float)
N = (nx-2)*(ny-2)*(nz-2)
c_vec = cc.reshape(N, 1)

#s_loc = int((nx-2)*(ny-2)*(nz-2)//2+(nx-2)*(ny-2)//2);
ix = (nx - 2) // 2 
iy = (ny - 2) // 2 
iz = 8
s_loc = iz * (nx - 2) * (ny - 2) + iy * (nx - 2) + ix
bzeros = np.zeros((nx-2, ny-2, nz-2), dtype=complex)
bb = source_ofd(f, f0, N, h, c_vec, s_loc).todense()
bzeros[(nx-2)//2, (ny-2)//2, 8] = bb[bb.nonzero()].real + 1j * bb[bb.nonzero()].imag

# A = matrix_ofd4(nx, nz, h, f, delta, c_vec)
A = matrix3fd4(nx, ny, nz, h, h, h, f, delta, c_vec)
b = bzeros.reshape(N, 1)
Gf = np.min(cc)/(2.5*f0*h)
print(Gf)

**Set random number seed**

In [ ]:
import random
import numpy as np
import torch
import tensorflow as tf
random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1234)
    torch.cuda.manual_seed_all(1234)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

tf.random.set_seed(1234)

**Construction of the training dataset (random version)**

In [ ]:
def generate_random_data(num_samples, impedance_matrix, nx, ny, nz, seed=1234):
    """
    Generate random complex-valued input-output training data for supervised learning.

    Parameters:
    -----------
    num_samples : int
        Number of training samples to generate.
    impedance_matrix : scipy.sparse matrix or ndarray
        The complex-valued system matrix A (e.g., impedance matrix) with shape (N, N),
        where N is the spatial grid size in one direction (assumes square grid).

    Returns:
    --------
    x_tensor : torch.Tensor
        Input tensor of shape (num_samples, 2, N, N), where the 2 channels represent
        the real and imaginary parts of a random complex input.
    y_tensor : torch.Tensor
        Output tensor of shape (num_samples, 2, N, N), representing the real and
        imaginary parts of the corresponding matrix-vector product A @ x.
    """
    x_data = np.zeros((num_samples, 2, nx - 2, ny - 2, nz - 2), dtype=float)    # Real and imaginary parts of input
    y_data = np.zeros((num_samples, 2, nx - 2, ny - 2, nz - 2), dtype=float)    # Real and imaginary parts of output
    np.random.seed(seed)
    for i in range(num_samples):
        # Generate random complex input field x with real and imaginary parts in [-1, 1]
        x_real = 2 * np.random.rand(nx - 2, ny - 2, nz - 2) - 1
        x_imag = 2 * np.random.rand(nx - 2, ny - 2, nz - 2) - 1
        x_data[i, 0, :, :, :] = x_real
        x_data[i, 1, :, :, :] = x_imag

        x_complex = (x_real + 1j * x_imag).reshape(-1, 1)  # Flatten for matrix-vector multiplication

        # Compute matrix-vector product: y = A @ x
        y_complex = impedance_matrix @ x_complex

        # Reshape result back to 2D grid and split into real/imaginary parts
        y_data[i, 0, :, :, :] = y_complex.real.reshape(nx - 2, ny - 2, nz - 2)
        y_data[i, 1, :, :, :] = y_complex.imag.reshape(nx - 2, ny - 2, nz - 2)

        # Normalize both x and y by maximum absolute value of y (to prevent scale explosion)
        max_val = np.max(np.abs(y_data[i]))
        x_data[i] /= max_val
        y_data[i] /= max_val

    # Convert to PyTorch tensors
    x_tensor = torch.tensor(x_data, dtype=torch.float32)
    y_tensor = torch.tensor(y_data, dtype=torch.float32)

    return x_tensor, y_tensor

**Constructing training dataset and validation dataset**

In [ ]:
# Generate training and validation datasets from the impedance matrix
trainX, trainY = generate_random_data(num_samples=20, impedance_matrix=A, nx=nx, ny=ny, nz=nz)
valX, valY = generate_random_data(num_samples=20, impedance_matrix=A, nx=nx, ny=ny, nz=nz)

# Print the shape of the generated datasets
print("Training input shape:", trainX.shape)
print("Training label shape:", trainY.shape)
print("Validation input shape:", valX.shape)
print("Validation label shape:", valY.shape)

# Check numerical accuracy of matrix-vector multiplication for a sample
sample_index = 2  # Index of the sample to validate
x_complex = (trainX[sample_index, 0, :, :, :] + 1j * trainX[sample_index, 1, :, :, :]).numpy()
y_complex = (trainY[sample_index, 0, :, :, :] + 1j * trainY[sample_index, 1, :, :, :]).numpy()

# Flatten for multiplication: A @ x should give y
x_flat = x_complex.reshape(-1, 1)
y_flat = y_complex.reshape(-1, 1)
error = A @ x_flat - y_flat

# Compute the norm of the error to verify consistency
print("Matrix-vector multiplication residual norm:", np.linalg.norm(error))

**Creating dataloader for batch training**

In [ ]:
# ================================
# Prepare PyTorch DataLoaders
# ================================
# Construct training dataset: input is the solution (y), output is the source (x)
train_dataset = TensorDataset(trainY, trainX)
# Construct validation dataset: same format as training
val_dataset = TensorDataset(valY, valX)
# Define training DataLoader with batch shuffling enabled
train_loader = DataLoader(
    dataset=train_dataset,   # Dataset containing training samples
    batch_size=32,           # Number of samples per batch
    shuffle=True             # Shuffle the data at every epoch
)
# Define validation DataLoader with no shuffling
val_loader = DataLoader(
    dataset=val_dataset,     # Dataset containing validation samples
    batch_size=32,           # Same batch size for consistency
    shuffle=False            # No need to shuffle validation data
)

**Network structure of the generator**

In [ ]:
# 基本卷积块
class Conv(nn.Module):
    def __init__(self, C_in, C_out):
        super(Conv, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv3d(C_in, C_out, 3, 1, 1),
            nn.Tanh(),
            nn.Conv3d(C_out, C_out, 3, 1, 1),
            nn.Tanh(),
        )
    def forward(self, x):
        return self.layer(x)

# 下采样模块
class DownSampling(nn.Module):
    def __init__(self, C):
        super(DownSampling, self).__init__()
        self.Down = nn.Sequential(
            # 使用卷积进行2倍的下采样，通道数不变
            nn.Conv3d(C, C, 3, 2, 1),
            nn.Tanh()
        )
    def forward(self, x):
        return self.Down(x)

# 上采样模块
class UpSampling(nn.Module):
    def __init__(self, C):
        super(UpSampling, self).__init__()
        # 特征图大小扩大2倍，通道数减半
        self.Up = nn.Conv3d(C, C // 2, 1, 1)

    def forward(self, x, r):
        # 使用邻近插值进行下采样
        up = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.Up(up)
        # 拼接，当前上采样的，和之前下采样过程中的
        return torch.cat((x, r), 1)
    
# 主干网络
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        # 4次下采样
        self.C1 = Conv(2, 64)
        self.D1 = DownSampling(64)
        self.C2 = Conv(64, 128)
        self.D2 = DownSampling(128)
        self.C3 = Conv(128, 256)
        self.D3 = DownSampling(256)
        self.C4 = Conv(256, 512)
        self.D4 = DownSampling(512)
        self.C5 = Conv(512, 1024)
        # 4次上采样
        self.U1 = UpSampling(1024)
        self.C6 = Conv(1024, 512)
        self.U2 = UpSampling(512)
        self.C7 = Conv(512, 256)
        self.U3 = UpSampling(256)
        self.C8 = Conv(256, 128)
        self.U4 = UpSampling(128)
        self.C9 = Conv(128, 64)

        self.Th = torch.nn.Tanh()
        self.pred = torch.nn.Conv3d(64, 2, 3, 1, 1)

    def forward(self, x):
        # 下采样部分
        R1 = self.C1(x)
        R2 = self.C2(self.D1(R1))
        R3 = self.C3(self.D2(R2))
        R4 = self.C4(self.D3(R3))
        Y1 = self.C5(self.D4(R4))
        # 上采样部分
        O1 = self.C6(self.U1(Y1, R4))
        O2 = self.C7(self.U2(O1, R3))
        O3 = self.C8(self.U3(O2, R2))
        O4 = self.C9(self.U4(O3, R1))
        return self.Th(self.pred(O4))

Network structure of the discriminator

In [ ]:
import torch
import torch.nn as nn

class PatchGAN3D(nn.Module):
    def __init__(self):
        super(PatchGAN3D, self).__init__()
        self.model = nn.Sequential(
            nn.Conv3d(4, 64, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
            
            nn.Conv3d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.Tanh(),
            
            nn.Conv3d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.Tanh(),
            
            nn.Conv3d(256, 512, kernel_size=4, stride=2, padding=1),  
            nn.BatchNorm3d(512),
            nn.Tanh(),
            
            nn.Conv3d(512, 1024, kernel_size=4, stride=1, padding=1),  
            nn.Tanh(),
            
            nn.Conv3d(1024, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x, y):
        x = torch.cat([x, y], dim=1)  
        x = self.model(x)
        return x.view(x.size(0), -1) 


**Network training**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Create the model
netD = PatchGAN3D().to(device)
netG = UNet().to(device)
optimizerD = optim.Adam(netD.parameters(), lr=1e-4,betas=(0.5,0.999))
optimizerG = optim.AdamW(netG.parameters(), lr=1e-5)
scheduler = StepLR(optimizerG, step_size=50, gamma=0.5)

BCE_Loss = nn.BCEWithLogitsLoss()
L1_Loss = nn.L1Loss()
L2_Loss = nn.MSELoss()

# 训练GAN
num_epochs = 100

# Initialize lists to monitor loss and accuracy
d_losses = []
g_losses = []
L1es=[]
val_d_losses1 = []
val_d_losses2 = []
val_g_losses1 = []
val_g_losses2 = []
for epoch in range(num_epochs):
    for i, (yi, xi) in enumerate(train_loader):
        xi = xi.float().to(device)
        yi = yi.float().to(device)
        #for ii in range(1):
            # Train Discriminator
        optimizerD.zero_grad()
        D_real = netD(xi, yi)
        D_real_label = torch.ones_like(D_real)
        D_real_loss = BCE_Loss(D_real, D_real_label)
        x_fake = netG(yi)
        D_fake = netD(x_fake, yi).detach()
        D_fake_loss = BCE_Loss(D_fake, torch.zeros_like(D_fake))
        D_loss = (D_real_loss + D_fake_loss) / 2
        D_loss.backward()
        optimizerD.step()

        #for ii in range(1):
            # Train Generator
        optimizerG.zero_grad()
        D_fake = netD(x_fake, yi)
        G_fake_loss = BCE_Loss(D_fake, torch.ones_like(D_fake))
        L1 = L1_Loss(xi, x_fake)
        G_loss = G_fake_loss*0.5 + L1*0.5
        G_loss.backward()
        optimizerG.step()
    scheduler.step()

     # 验证
    val_d_loss1 = 0.0
    val_d_loss2 = 0.0
    val_g_loss1 = 0.0
    val_g_loss2 = 0.0
    with torch.no_grad():
        for val_yi, val_xi in val_loader:
            val_xi = val_xi.float().to(device)
            val_yi = val_yi.float().to(device)
            val_x_fake = netG(val_yi)
            val_d_loss1 += BCE_Loss(netD(val_xi, val_yi), torch.ones_like(netD(val_xi, val_yi))).item()
            val_d_loss2 += BCE_Loss(netD(val_x_fake, val_yi), torch.zeros_like(netD(val_xi, val_yi))).item()
            val_g_loss1 += BCE_Loss(netD(val_x_fake, val_yi), torch.ones_like(netD(val_xi, val_yi))).item()
            val_g_loss2 += L1_Loss(val_xi, val_x_fake).item()
        val_d_loss1 /= len(val_loader)
        val_d_loss2 /= len(val_loader)
        val_g_loss1 /= len(val_loader)
        val_g_loss2 /= len(val_loader)
        val_d_losses1.append(val_d_loss1)
        val_d_losses2.append(val_d_loss2)
        val_g_losses1.append(val_g_loss1)
        val_g_losses2.append(val_g_loss2)

    d_losses.append(D_loss.item())
    g_losses.append(G_loss.item())
    L1es.append(L1.item())
    print(f"Epoch [{epoch + 1}/{num_epochs}], d_loss: {D_loss.item()}, g_loss: {G_loss.item()},L1:{L1}")
    print(f"Epoch [{epoch + 1}/{num_epochs}], val_d_loss1: {val_d_loss1}, val_d_loss2: {val_d_loss2},val_g_loss1: {val_g_loss1}, val_g_loss2: {val_g_loss2}")

**Save the trained network parameters**

In [ ]:
torch.save(netG.state_dict(), 'netG_weights.pth') 